# Pruning Experiment Visualization

This notebook visualizes results from `simulPrune.py` (CNN) and `simulPruneVit.py` (ViT) experiments.

**Metrics visualized:**
- Training Energy (kWh)
- AUC (Area Under ROC Curve)
- Accuracy
- Model Size (MB)

All plots include error bars (standard deviation across trials).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('colorblind')
plt.rcParams['figure.dpi'] = 100

## Configuration

**Set the paths to your CSV files below:**

In [ ]:
# ============================================================
# SET YOUR CSV FILE PATHS HERE
# ============================================================

# Path to all_experiments_combined.csv (contains individual trial data)
EXPERIMENTS_CSV_PATH = "path/to/all_experiments_combined.csv"

# Path to all_datasets_summary.csv (contains aggregated statistics)
SUMMARY_CSV_PATH = "path/to/all_datasets_summary.csv"

## Load Data

In [ ]:
def load_csv_safe(path, name):
    """Load CSV with error handling"""
    try:
        df = pd.read_csv(path)
        print(f"Loaded {name}: {len(df)} rows, {len(df.columns)} columns")
        return df
    except FileNotFoundError:
        print(f"File not found: {path}")
        print(f"Please update the path variable.")
        return None
    except Exception as e:
        print(f"Error loading {path}: {e}")
        return None

experiments_df = load_csv_safe(EXPERIMENTS_CSV_PATH, "experiments")
summary_df = load_csv_safe(SUMMARY_CSV_PATH, "summary")

In [ ]:
# Preview the data
if experiments_df is not None:
    print("=== Experiments Data ===")
    print(f"Columns: {list(experiments_df.columns)}")
    display(experiments_df.head())

if summary_df is not None:
    print("\n=== Summary Data ===")
    print(f"Columns: {list(summary_df.columns)}")
    display(summary_df.head())

## Plotting Functions

In [ ]:
def find_column(df, candidates):
    """Find first matching column from candidates list"""
    if df is None:
        return None
    for col in candidates:
        if col in df.columns:
            return col
    return None

def plot_grouped_bars(df, value_col, ylabel, title, err_col=None, figsize=(12, 6)):
    """
    Create grouped bar chart with methods on x-axis and datasets as groups.
    Computes mean/std from raw data if err_col not provided.
    """
    if df is None:
        print("No data available.")
        return None, None
    
    if value_col not in df.columns:
        print(f"Column '{value_col}' not found. Available: {list(df.columns)}")
        return None, None
    
    # Determine grouping
    has_method = 'method' in df.columns
    has_dataset = 'dataset' in df.columns
    has_model = 'model_name' in df.columns
    
    # Aggregate if needed (raw trial data)
    if err_col is None or err_col not in df.columns:
        group_cols = [c for c in ['method', 'dataset', 'model_name'] if c in df.columns]
        if group_cols:
            agg_df = df.groupby(group_cols)[value_col].agg(['mean', 'std']).reset_index()
            agg_df.columns = group_cols + ['mean', 'std']
            plot_df = agg_df
            val_col = 'mean'
            std_col = 'std'
        else:
            plot_df = df
            val_col = value_col
            std_col = None
    else:
        plot_df = df
        val_col = value_col
        std_col = err_col
    
    fig, ax = plt.subplots(figsize=figsize)
    
    x_col = 'method' if has_method else ('dataset' if has_dataset else 'model_name')
    hue_col = 'dataset' if has_dataset and x_col != 'dataset' else ('model_name' if has_model and x_col != 'model_name' else None)
    
    x_labels = plot_df[x_col].unique()
    x = np.arange(len(x_labels))
    
    if hue_col and hue_col in plot_df.columns:
        hue_labels = plot_df[hue_col].unique()
        width = 0.8 / len(hue_labels)
        colors = plt.cm.Set2(np.linspace(0, 1, len(hue_labels)))
        
        for i, hue in enumerate(hue_labels):
            subset = plot_df[plot_df[hue_col] == hue]
            values = [subset[subset[x_col] == xl][val_col].values[0] if len(subset[subset[x_col] == xl]) > 0 else 0 for xl in x_labels]
            errors = [subset[subset[x_col] == xl][std_col].values[0] if std_col and len(subset[subset[x_col] == xl]) > 0 else 0 for xl in x_labels]
            errors = [0 if pd.isna(e) else e for e in errors]
            
            offset = (i - len(hue_labels)/2 + 0.5) * width
            ax.bar(x + offset, values, width, label=hue, yerr=errors, capsize=3,
                   color=colors[i], edgecolor='black', linewidth=0.5)
        
        ax.legend(title=hue_col.replace('_', ' ').title(), bbox_to_anchor=(1.02, 1), loc='upper left')
    else:
        values = plot_df[val_col].values
        errors = plot_df[std_col].values if std_col else np.zeros_like(values)
        errors = [0 if pd.isna(e) else e for e in errors]
        colors = plt.cm.Set2(np.linspace(0, 1, len(x_labels)))
        ax.bar(x, values, 0.6, yerr=errors, capsize=3, color=colors, edgecolor='black', linewidth=0.5)
    
    ax.set_xlabel(x_col.replace('_', ' ').title(), fontsize=12)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(x_labels, rotation=45, ha='right')
    plt.tight_layout()
    
    return fig, ax

In [ ]:
def create_dashboard(df, metrics, suptitle, figsize=(16, 12)):
    """
    Create 2x2 dashboard of metrics.
    
    metrics: list of dicts with keys 'col', 'err_col' (optional), 'label', 'title'
    """
    if df is None:
        print("No data available.")
        return None, None
    
    fig, axes = plt.subplots(2, 2, figsize=figsize)
    axes = axes.flatten()
    
    has_method = 'method' in df.columns
    has_dataset = 'dataset' in df.columns
    has_model = 'model_name' in df.columns
    
    x_col = 'method' if has_method else ('dataset' if has_dataset else 'model_name')
    hue_col = 'dataset' if has_dataset and x_col != 'dataset' else ('model_name' if has_model and x_col != 'model_name' else None)
    
    for idx, m in enumerate(metrics[:4]):
        ax = axes[idx]
        val_col = m['col']
        err_col = m.get('err_col')
        
        if val_col not in df.columns:
            ax.text(0.5, 0.5, f"'{val_col}' not found", ha='center', va='center', transform=ax.transAxes)
            ax.set_title(m['title'])
            continue
        
        # Aggregate if no err_col
        if err_col is None or err_col not in df.columns:
            group_cols = [c for c in ['method', 'dataset', 'model_name'] if c in df.columns]
            if group_cols:
                plot_df = df.groupby(group_cols)[val_col].agg(['mean', 'std']).reset_index()
                plot_df.columns = group_cols + ['mean', 'std']
                use_val = 'mean'
                use_err = 'std'
            else:
                plot_df = df
                use_val = val_col
                use_err = None
        else:
            plot_df = df
            use_val = val_col
            use_err = err_col
        
        x_labels = plot_df[x_col].unique()
        x = np.arange(len(x_labels))
        
        if hue_col and hue_col in plot_df.columns:
            hue_labels = plot_df[hue_col].unique()
            width = 0.8 / len(hue_labels)
            colors = plt.cm.Set2(np.linspace(0, 1, len(hue_labels)))
            
            for i, hue in enumerate(hue_labels):
                subset = plot_df[plot_df[hue_col] == hue]
                values = [subset[subset[x_col] == xl][use_val].values[0] if len(subset[subset[x_col] == xl]) > 0 else 0 for xl in x_labels]
                errors = [subset[subset[x_col] == xl][use_err].values[0] if use_err and len(subset[subset[x_col] == xl]) > 0 else 0 for xl in x_labels]
                errors = [0 if pd.isna(e) else e for e in errors]
                
                offset = (i - len(hue_labels)/2 + 0.5) * width
                ax.bar(x + offset, values, width, label=hue, yerr=errors, capsize=2,
                       color=colors[i], edgecolor='black', linewidth=0.5)
            
            if idx == 1:
                ax.legend(title=hue_col.replace('_', ' ').title(), fontsize=8)
        else:
            values = [plot_df[plot_df[x_col] == xl][use_val].values[0] if len(plot_df[plot_df[x_col] == xl]) > 0 else 0 for xl in x_labels]
            errors = [plot_df[plot_df[x_col] == xl][use_err].values[0] if use_err and len(plot_df[plot_df[x_col] == xl]) > 0 else 0 for xl in x_labels]
            errors = [0 if pd.isna(e) else e for e in errors]
            colors = plt.cm.Set2(np.linspace(0, 1, len(x_labels)))
            ax.bar(x, values, 0.6, yerr=errors, capsize=3, color=colors, edgecolor='black', linewidth=0.5)
        
        ax.set_xlabel(x_col.replace('_', ' ').title(), fontsize=10)
        ax.set_ylabel(m['label'], fontsize=10)
        ax.set_title(m['title'], fontsize=11, fontweight='bold')
        ax.set_xticks(x)
        ax.set_xticklabels(x_labels, rotation=45, ha='right', fontsize=9)
    
    fig.suptitle(suptitle, fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    return fig, axes

## Visualize all_experiments_combined.csv

Individual trial data - mean and std computed automatically.

In [ ]:
if experiments_df is not None:
    # Auto-detect columns
    energy_col = find_column(experiments_df, ['training_energy_kwh', 'energy_kwh', 'train_energy_kwh'])
    auc_col = find_column(experiments_df, ['test_auc', 'auc', 'val_auc'])
    acc_col = find_column(experiments_df, ['test_acc', 'accuracy', 'acc', 'val_acc'])
    size_col = find_column(experiments_df, ['model_size_mb', 'size_mb', 'model_size'])
    
    print(f"Detected columns: energy={energy_col}, auc={auc_col}, acc={acc_col}, size={size_col}")

In [ ]:
# Individual plots
if experiments_df is not None and energy_col:
    plot_grouped_bars(experiments_df, energy_col, 'Energy (kWh)', 'Training Energy Comparison')
    plt.show()

In [ ]:
if experiments_df is not None and auc_col:
    plot_grouped_bars(experiments_df, auc_col, 'AUC', 'AUC Comparison')
    plt.show()

In [ ]:
if experiments_df is not None and acc_col:
    plot_grouped_bars(experiments_df, acc_col, 'Accuracy', 'Accuracy Comparison')
    plt.show()

In [ ]:
if experiments_df is not None and size_col:
    plot_grouped_bars(experiments_df, size_col, 'Model Size (MB)', 'Model Size Comparison')
    plt.show()

In [ ]:
# Combined Dashboard
if experiments_df is not None:
    metrics = [
        {'col': energy_col or 'training_energy_kwh', 'label': 'Energy (kWh)', 'title': 'Training Energy'},
        {'col': auc_col or 'test_auc', 'label': 'AUC', 'title': 'Area Under ROC Curve'},
        {'col': acc_col or 'test_acc', 'label': 'Accuracy', 'title': 'Test Accuracy'},
        {'col': size_col or 'model_size_mb', 'label': 'Size (MB)', 'title': 'Model Size'},
    ]
    create_dashboard(experiments_df, metrics, 'Experiments Combined: Method Comparison')
    plt.show()

## Visualize all_datasets_summary.csv

Pre-computed statistics with _mean and _std columns.

In [ ]:
if summary_df is not None:
    # Auto-detect columns (with _mean suffix)
    energy_mean = find_column(summary_df, ['training_energy_kwh_mean', 'energy_kwh_mean'])
    auc_mean = find_column(summary_df, ['test_auc_mean', 'auc_mean'])
    acc_mean = find_column(summary_df, ['test_acc_mean', 'accuracy_mean', 'acc_mean'])
    size_mean = find_column(summary_df, ['model_size_mb_mean', 'size_mb_mean'])
    
    print(f"Detected columns: energy={energy_mean}, auc={auc_mean}, acc={acc_mean}, size={size_mean}")

In [ ]:
# Individual plots with error bars from _std columns
if summary_df is not None and energy_mean:
    energy_std = energy_mean.replace('_mean', '_std')
    plot_grouped_bars(summary_df, energy_mean, 'Energy (kWh)', 'Training Energy (Summary)', err_col=energy_std)
    plt.show()

In [ ]:
if summary_df is not None and auc_mean:
    auc_std = auc_mean.replace('_mean', '_std')
    plot_grouped_bars(summary_df, auc_mean, 'AUC', 'AUC (Summary)', err_col=auc_std)
    plt.show()

In [ ]:
if summary_df is not None and acc_mean:
    acc_std = acc_mean.replace('_mean', '_std')
    plot_grouped_bars(summary_df, acc_mean, 'Accuracy', 'Accuracy (Summary)', err_col=acc_std)
    plt.show()

In [ ]:
if summary_df is not None and size_mean:
    size_std = size_mean.replace('_mean', '_std')
    plot_grouped_bars(summary_df, size_mean, 'Model Size (MB)', 'Model Size (Summary)', err_col=size_std)
    plt.show()

In [ ]:
# Combined Dashboard for Summary
if summary_df is not None:
    metrics = [
        {'col': energy_mean or 'training_energy_kwh_mean', 'err_col': (energy_mean or '').replace('_mean', '_std'), 'label': 'Energy (kWh)', 'title': 'Training Energy'},
        {'col': auc_mean or 'test_auc_mean', 'err_col': (auc_mean or '').replace('_mean', '_std'), 'label': 'AUC', 'title': 'Area Under ROC Curve'},
        {'col': acc_mean or 'test_acc_mean', 'err_col': (acc_mean or '').replace('_mean', '_std'), 'label': 'Accuracy', 'title': 'Test Accuracy'},
        {'col': size_mean or 'model_size_mb_mean', 'err_col': (size_mean or '').replace('_mean', '_std'), 'label': 'Size (MB)', 'title': 'Model Size'},
    ]
    create_dashboard(summary_df, metrics, 'Datasets Summary: Method Comparison')
    plt.show()

## Per-Dataset Breakdown

In [ ]:
def plot_per_dataset(df, val_col, ylabel, title, err_col=None, figsize=(8, 5)):
    """Create a separate figure for each dataset, comparing methods as bars"""
    if df is None or 'dataset' not in df.columns:
        print("No data or 'dataset' column not found.")
        return

    if val_col not in df.columns:
        print(f"Column '{val_col}' not found.")
        return

    datasets = df['dataset'].unique()
    has_trial = 'trial' in df.columns
    figures = []

    for ds in datasets:
        fig, ax = plt.subplots(figsize=figsize)
        subset = df[df['dataset'] == ds]

        if has_trial and (err_col is None or err_col not in df.columns):
            agg = subset.groupby('method')[val_col].agg(['mean', 'std']).reset_index()
            methods = agg['method'].values
            values = agg['mean'].values
            errors = agg['std'].fillna(0).values
        else:
            methods = subset['method'].values
            values = subset[val_col].values
            errors = subset[err_col].fillna(0).values if err_col and err_col in subset.columns else np.zeros_like(values)

        x = np.arange(len(methods))
        colors = plt.cm.Set2(np.linspace(0, 1, len(methods)))
        ax.bar(x, values, yerr=errors, capsize=4, color=colors, edgecolor='black', linewidth=0.5)
        ax.set_title(f"{title} - {ds}", fontsize=12, fontweight='bold')
        ax.set_xlabel('Method', fontsize=10)
        ax.set_ylabel(ylabel, fontsize=10)
        ax.set_xticks(x)
        ax.set_xticklabels(methods, rotation=45, ha='right', fontsize=9)
        plt.tight_layout()
        figures.append((fig, ax))
        plt.show()

    return figures

In [ ]:
# Per-dataset from experiments
if experiments_df is not None and 'dataset' in experiments_df.columns and acc_col:
    plot_per_dataset(experiments_df, acc_col, 'Accuracy', 'Accuracy by Dataset')

In [ ]:
# Per-dataset from summary
if summary_df is not None and 'dataset' in summary_df.columns and acc_mean:
    acc_std = acc_mean.replace('_mean', '_std')
    plot_per_dataset(summary_df, acc_mean, 'Accuracy', 'Accuracy by Dataset (Summary)', err_col=acc_std)

In [ ]:
# All metrics per-dataset from experiments
if experiments_df is not None and 'dataset' in experiments_df.columns:
    if energy_col:
        plot_per_dataset(experiments_df, energy_col, 'Energy (kWh)', 'Training Energy')
    if auc_col:
        plot_per_dataset(experiments_df, auc_col, 'AUC', 'AUC')
    if size_col:
        plot_per_dataset(experiments_df, size_col, 'Model Size (MB)', 'Model Size')

In [ ]:
# All metrics per-dataset from summary
if summary_df is not None and 'dataset' in summary_df.columns:
    if energy_mean:
        plot_per_dataset(summary_df, energy_mean, 'Energy (kWh)', 'Training Energy (Summary)', 
                         err_col=energy_mean.replace('_mean', '_std'))
    if auc_mean:
        plot_per_dataset(summary_df, auc_mean, 'AUC', 'AUC (Summary)', 
                         err_col=auc_mean.replace('_mean', '_std'))
    if size_mean:
        plot_per_dataset(summary_df, size_mean, 'Model Size (MB)', 'Model Size (Summary)', 
                         err_col=size_mean.replace('_mean', '_std'))

## Save Figures (Optional)

In [ ]:
SAVE_FIGURES = False  # Set True to save
SAVE_DIR = "./figures"

if SAVE_FIGURES:
    import os
    os.makedirs(SAVE_DIR, exist_ok=True)
    
    if experiments_df is not None:
        fig, _ = create_dashboard(experiments_df, metrics, 'Experiments: Method Comparison')
        if fig:
            fig.savefig(f"{SAVE_DIR}/experiments_dashboard.png", dpi=300, bbox_inches='tight')
            print(f"Saved: {SAVE_DIR}/experiments_dashboard.png")
    
    if summary_df is not None:
        fig, _ = create_dashboard(summary_df, metrics, 'Summary: Method Comparison')
        if fig:
            fig.savefig(f"{SAVE_DIR}/summary_dashboard.png", dpi=300, bbox_inches='tight')
            print(f"Saved: {SAVE_DIR}/summary_dashboard.png")